# Knowledge Graph Memory — Structured Memory with Graphs

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/agents/12_knowledge_graph_memory.ipynb)

## What This Notebook Teaches You

Vector stores (Notebook 11) find semantically similar memories, but they **lose structure**. If an agent learns "Alice manages the backend team" and "The backend team owns the payment service," a vector store can find each fact individually — but it can't answer "Who is responsible for the payment service?" because that requires **traversing a relationship chain**.

Knowledge graphs solve this by storing information as **entities** and **relationships**, enabling multi-hop reasoning.

By the end, you will:

1. **Understand why graphs complement vectors** — different query types, different strengths
2. **Extract entity-relation triples** from text using an LLM
3. **Build a GraphMemory class** with networkx — add, query, multi-hop, and visualize
4. **Build a graph-enhanced agent** that stores and queries a knowledge graph
5. **Compare vector vs. graph memory** on semantic vs. relational queries
6. **Combine both into a hybrid memory** that routes queries appropriately

---

> **Prerequisites**: Notebooks 10–11 (short-term and long-term memory).
> **Runtime**: Google Colab with T4 GPU.
> **Time**: ~45–60 minutes.

## Part 0: Environment Setup

We need the LLM, embedding model, FAISS, **and** networkx for graph operations.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy networkx

import torch
import time
import json
import re
import math
import numpy as np
import faiss
import networkx as nx
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Set, Union
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype="auto",
)
embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5", cache_folder=CACHE_DIR)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

def embed_texts(texts):
    return embed_model.encode(texts, normalize_embeddings=True)

print(f"✓ LLM: {MODEL_NAME} | GPU: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"✓ Embeddings: BAAI/bge-base-en-v1.5 | Graph: networkx")

## 1. Why Graphs? When Vector Stores Aren't Enough

### The Relationship Problem

Consider these facts an agent has learned:

1. *"Alice is the tech lead of Project Phoenix"*
2. *"Project Phoenix uses PostgreSQL for its database"*
3. *"PostgreSQL 15 introduced MERGE statement support"*

**Semantic search** (vector store) can find each fact individually. But it struggles with:

- **Multi-hop queries**: "What database feature is available to Alice's project?" (requires: Alice → Phoenix → PostgreSQL → MERGE)
- **Relationship queries**: "Who works on what?" (requires traversing person→project edges)
- **Structural queries**: "What projects use PostgreSQL?" (requires reverse traversal)

**Knowledge graphs** store information as **(subject, relation, object) triples** and support traversal:

```
Alice ──[tech_lead_of]──▶ Project Phoenix ──[uses_database]──▶ PostgreSQL ──[supports]──▶ MERGE
```

| Query Type | Vector Store | Knowledge Graph |
|-----------|-------------|-----------------|
| "Tell me about PostgreSQL" | ✅ Semantic match | ⚠️ Needs entity lookup |
| "What database does Phoenix use?" | ⚠️ May miss relationship | ✅ Direct edge traversal |
| "What projects does Alice lead?" | ⚠️ Depends on embedding | ✅ Direct edge traversal |
| "What DB features are available to Alice?" | ❌ Multi-hop needed | ✅ 3-hop traversal |

In [ ]:
# Quick demonstration of the relationship problem with embeddings
facts = [
    "Alice is the tech lead of Project Phoenix",
    "Project Phoenix uses PostgreSQL for its database",
    "PostgreSQL 15 introduced the MERGE statement",
    "Bob is the designer on Project Phoenix",
    "Project Phoenix launches in Q2 2025",
]

fact_embeddings = embed_texts(facts)

# Query that requires multi-hop reasoning
queries = [
    "What database features are available to Alice's project?",
    "Who works on the project that uses PostgreSQL?",
    "When does the PostgreSQL project launch?",
]

print("Vector Search vs. Relational Queries")
print("=" * 70)

for query in queries:
    query_emb = embed_texts([query])[0]
    similarities = [float(np.dot(query_emb, f)) for f in fact_embeddings]
    ranked = sorted(zip(similarities, facts), reverse=True)

    print(f"\n🔍 Query: '{query}'")
    print(f"   Top results:")
    for sim, fact in ranked[:3]:
        print(f"   [{sim:.3f}] {fact}")
    print(f"   ⚠️  Vector search finds related facts but can't chain them into an answer")

print("\n" + "=" * 70)
print("A knowledge graph would connect: Alice → Phoenix → PostgreSQL → MERGE")
print("and answer all these queries via graph traversal.")

## 2. Entity-Relation Extraction

The first step in building a knowledge graph is **extracting structured triples** from unstructured text. We use the LLM to identify:

- **Subject**: The entity performing or being described
- **Relation**: The relationship type (normalized to lowercase_with_underscores)
- **Object**: The entity being acted upon or related to

This is a classic NLP task, but LLMs make it surprisingly easy with the right prompt.

In [ ]:
def extract_triples(text, max_triples=10):
    """Extract (subject, relation, object) triples from text using the LLM."""
    messages = [
        {"role": "system", "content": """You are an expert at extracting structured knowledge from text.
Extract (subject, relation, object) triples from the given text.

Rules:
- Normalize entity names (e.g., "Dr. Smith" → "Dr Smith")
- Use lowercase_snake_case for relations (e.g., "works_at", "is_a", "manages")
- Extract concrete facts, not opinions or speculation
- One triple per line in format: subject | relation | object
- Output ONLY triples, no commentary"""},
        {"role": "user", "content": f"Extract triples from:\n{text}"}
    ]

    response = generate(messages, max_new_tokens=300, temperature=0.2)

    triples = []
    for line in response.strip().split("\n"):
        line = line.strip().lstrip("- •0123456789.")
        parts = [p.strip() for p in line.split("|")]
        if len(parts) == 3 and all(p for p in parts):
            subject, relation, obj = parts
            triples.append((subject, relation, obj))

    return triples[:max_triples]

# Test extraction on sample texts
test_texts = [
    "Alice is the tech lead of Project Phoenix. The project uses PostgreSQL for its database and Redis for caching. Bob handles the frontend design.",
    "Dr. Sarah Chen published a paper on transformer architectures in 2023. She works at Stanford University and collaborates with Prof. James Lee from MIT.",
    "The FastAPI application is deployed on AWS ECS. It uses Fargate for container management and CloudWatch for monitoring. The deployment pipeline runs through GitHub Actions.",
]

print("Entity-Relation Extraction")
print("=" * 70)

for i, text in enumerate(test_texts, 1):
    print(f"\n📝 Text {i}: {text[:80]}...")
    triples = extract_triples(text)
    print(f"   Extracted {len(triples)} triples:")
    for s, r, o in triples:
        print(f"   ({s}) ──[{r}]──▶ ({o})")

## 3. GraphMemory Class

Now we build the core `GraphMemory` class using networkx. It supports:

- `add_memory(text)` — extract triples from text and add to graph
- `add_triple(s, r, o)` — add a single triple directly
- `query(question)` — find relevant entities and their connections
- `get_entity_context(entity)` — get all facts about an entity
- `multi_hop_query(entity, hops=2)` — traverse N hops from an entity
- `visualize()` — ASCII visualization of the graph

In [ ]:
class GraphMemory:
    """Knowledge graph memory using networkx for structured fact storage."""

    def __init__(self):
        self.graph = nx.DiGraph()
        self.entity_index = {}  # normalized name → canonical name
        self.triple_count = 0

    def _normalize(self, entity):
        """Normalize entity name for matching."""
        return entity.lower().strip()

    def _canonical(self, entity):
        """Get or create canonical name for an entity."""
        normalized = self._normalize(entity)
        if normalized not in self.entity_index:
            self.entity_index[normalized] = entity.strip()
        return self.entity_index[normalized]

    def add_triple(self, subject, relation, obj, metadata=None):
        """Add a single (subject, relation, object) triple to the graph."""
        s = self._canonical(subject)
        o = self._canonical(obj)
        r = relation.lower().strip()

        self.graph.add_node(s, type="entity")
        self.graph.add_node(o, type="entity")

        # Add edge with relation and metadata
        edge_data = {"relation": r, "added_at": time.time()}
        if metadata:
            edge_data.update(metadata)

        # Check for existing edge with same relation
        if self.graph.has_edge(s, o):
            existing = self.graph[s][o]
            if existing.get("relation") == r:
                return False  # duplicate
        
        self.graph.add_edge(s, o, **edge_data)
        self.triple_count += 1
        return True

    def add_memory(self, text, metadata=None):
        """Extract triples from text and add to graph."""
        triples = extract_triples(text)
        added = 0
        for s, r, o in triples:
            if self.add_triple(s, r, o, metadata):
                added += 1
        return added, triples

    def get_entity_context(self, entity):
        """Get all facts about an entity (incoming and outgoing edges)."""
        canonical = self._canonical(entity)
        if canonical not in self.graph:
            # Try fuzzy match
            normalized = self._normalize(entity)
            for node in self.graph.nodes():
                if normalized in self._normalize(node):
                    canonical = node
                    break
            else:
                return []

        facts = []

        # Outgoing edges: entity → relation → other
        for _, target, data in self.graph.out_edges(canonical, data=True):
            facts.append({
                "subject": canonical,
                "relation": data.get("relation", "related_to"),
                "object": target,
                "direction": "outgoing"
            })

        # Incoming edges: other → relation → entity
        for source, _, data in self.graph.in_edges(canonical, data=True):
            facts.append({
                "subject": source,
                "relation": data.get("relation", "related_to"),
                "object": canonical,
                "direction": "incoming"
            })

        return facts

    def multi_hop_query(self, entity, hops=2):
        """Traverse N hops from an entity, collecting all connected facts."""
        canonical = self._canonical(entity)
        if canonical not in self.graph:
            return {"entity": entity, "facts": [], "entities_reached": set()}

        visited = set()
        facts = []
        frontier = {canonical}

        for hop in range(hops):
            next_frontier = set()
            for node in frontier:
                if node in visited:
                    continue
                visited.add(node)

                # Outgoing
                for _, target, data in self.graph.out_edges(node, data=True):
                    facts.append({
                        "subject": node,
                        "relation": data.get("relation", "related_to"),
                        "object": target,
                        "hop": hop + 1
                    })
                    next_frontier.add(target)

                # Incoming
                for source, _, data in self.graph.in_edges(node, data=True):
                    facts.append({
                        "subject": source,
                        "relation": data.get("relation", "related_to"),
                        "object": node,
                        "hop": hop + 1
                    })
                    next_frontier.add(source)

            frontier = next_frontier - visited

        return {
            "entity": entity,
            "facts": facts,
            "entities_reached": visited,
            "hops": hops
        }

    def query(self, question):
        """Find relevant graph context for a natural language question."""
        # Extract likely entity names from question
        messages = [
            {"role": "system", "content": "Extract entity names from this question. Output one entity per line, nothing else."},
            {"role": "user", "content": question}
        ]
        response = generate(messages, max_new_tokens=100, temperature=0.2)

        entities = [e.strip().lstrip("- •") for e in response.strip().split("\n") if e.strip()]

        all_facts = []
        for entity in entities:
            context = self.get_entity_context(entity)
            all_facts.extend(context)

        # If no entities found, try matching against all nodes
        if not all_facts:
            q_lower = question.lower()
            for node in self.graph.nodes():
                if self._normalize(node) in q_lower or q_lower in self._normalize(node):
                    context = self.get_entity_context(node)
                    all_facts.extend(context)

        # Deduplicate
        seen = set()
        unique_facts = []
        for f in all_facts:
            key = (f["subject"], f["relation"], f["object"])
            if key not in seen:
                seen.add(key)
                unique_facts.append(f)

        return unique_facts

    def visualize(self, max_nodes=20):
        """ASCII visualization of the knowledge graph."""
        lines = []
        lines.append(f"Knowledge Graph: {self.graph.number_of_nodes()} entities, {self.graph.number_of_edges()} relations")
        lines.append("=" * 60)

        # Group by source entity
        shown = 0
        for node in sorted(self.graph.nodes()):
            out_edges = list(self.graph.out_edges(node, data=True))
            if out_edges:
                lines.append(f"\n  📌 {node}")
                for _, target, data in out_edges:
                    rel = data.get("relation", "?")
                    lines.append(f"     ──[{rel}]──▶ {target}")
                shown += 1
                if shown >= max_nodes:
                    lines.append(f"\n  ... ({self.graph.number_of_nodes() - shown} more entities)")
                    break

        return "\n".join(lines)

    def stats(self):
        return {
            "entities": self.graph.number_of_nodes(),
            "relations": self.graph.number_of_edges(),
            "triples_added": self.triple_count,
        }

    def __repr__(self):
        s = self.stats()
        return f"GraphMemory({s['entities']} entities, {s['relations']} relations)"


# Build a knowledge graph
gm = GraphMemory()

# Add knowledge about a software project
texts = [
    "Alice is the tech lead of Project Phoenix. She has 8 years of experience in Python and distributed systems.",
    "Project Phoenix uses PostgreSQL for its database and Redis for caching. The project is deployed on AWS ECS.",
    "Bob is the frontend developer on Project Phoenix. He specializes in React and TypeScript.",
    "PostgreSQL 15 supports the MERGE statement and improved logical replication. The database runs on RDS.",
    "AWS ECS uses Fargate for serverless container management. CloudWatch handles monitoring for ECS.",
    "The CI/CD pipeline for Project Phoenix runs on GitHub Actions with staging and production environments.",
]

print("Building Knowledge Graph from Text")
print("=" * 60)

for text in texts:
    added, triples = gm.add_memory(text)
    print(f"\n📝 '{text[:60]}...'")
    print(f"   Extracted {len(triples)} triples, added {added} new:")
    for s, r, o in triples:
        print(f"   ({s}) ──[{r}]──▶ ({o})")

print(f"\n{gm}")
print(f"\n{gm.visualize()}")

In [ ]:
# Test different query types on the knowledge graph

print("Graph Queries — Entity Context, Multi-Hop, and Natural Language")
print("=" * 70)

# 1. Entity context
print("\n1️⃣  Entity Context: 'Alice'")
facts = gm.get_entity_context("Alice")
for f in facts:
    print(f"   {f['subject']} ──[{f['relation']}]──▶ {f['object']}  ({f['direction']})")

# 2. Multi-hop query
print("\n2️⃣  Multi-Hop Query: 'Alice', 2 hops")
result = gm.multi_hop_query("Alice", hops=2)
print(f"   Entities reached: {result['entities_reached']}")
for f in result["facts"]:
    print(f"   [hop {f['hop']}] {f['subject']} ──[{f['relation']}]──▶ {f['object']}")

# 3. Natural language query
print("\n3️⃣  NL Query: 'What database does Project Phoenix use?'")
facts = gm.query("What database does Project Phoenix use?")
for f in facts:
    print(f"   {f['subject']} ──[{f['relation']}]──▶ {f['object']}")

print("\n4️⃣  NL Query: 'Who works on Project Phoenix?'")
facts = gm.query("Who works on Project Phoenix?")
for f in facts:
    print(f"   {f['subject']} ──[{f['relation']}]──▶ {f['object']}")

## 4. Graph-Enhanced Agent

Now we build an agent that maintains a knowledge graph as it conversations. The agent:

1. Extracts facts from conversations and stores them as triples
2. Queries the graph when answering questions
3. Can explain its reasoning by showing the graph traversal path

In [ ]:
class GraphAgent:
    """Agent that maintains and queries a knowledge graph."""

    SYSTEM_TEMPLATE = """You are a helpful assistant with a knowledge graph memory.

{graph_context}

When you learn new facts, say STORE: <fact as text>
When answering, reference specific facts from the graph when available.
Be precise and factual."""

    def __init__(self):
        self.graph = GraphMemory()
        self.conversation = []

    def step(self, user_input, verbose=True):
        """Process one user message with graph memory augmentation."""

        # 1. Query graph for relevant context
        graph_facts = self.graph.query(user_input)
        if graph_facts:
            context_lines = ["Known facts from knowledge graph:"]
            for f in graph_facts[:10]:
                context_lines.append(f"  - {f['subject']} {f['relation'].replace('_', ' ')} {f['object']}")
            graph_context = "\n".join(context_lines)
        else:
            graph_context = "Knowledge graph: (no relevant facts found)"

        # 2. Build prompt
        system = self.SYSTEM_TEMPLATE.format(graph_context=graph_context)
        messages = [{"role": "system", "content": system}]
        messages.extend(self.conversation[-10:])
        messages.append({"role": "user", "content": user_input})

        # 3. Generate
        response = generate(messages, max_new_tokens=300, temperature=0.7)

        # 4. Store conversation
        self.conversation.append({"role": "user", "content": user_input})
        self.conversation.append({"role": "assistant", "content": response})

        # 5. Extract and store new facts
        for line in response.split("\n"):
            line = line.strip()
            if line.startswith("STORE:"):
                fact_text = line[6:].strip()
                if fact_text:
                    added, triples = self.graph.add_memory(fact_text)
                    if verbose and triples:
                        print(f"   💾 Stored {added} new triples from: {fact_text[:50]}")

        # Also extract from user input if it contains factual info
        if any(kw in user_input.lower() for kw in ["is a", "works", "uses", "built", "manages", "leads", "deployed"]):
            added, triples = self.graph.add_memory(user_input)
            if verbose and added > 0:
                print(f"   💾 Auto-extracted {added} triples from user input")

        if verbose and graph_facts:
            print(f"   🔗 Retrieved {len(graph_facts)} facts from graph")

        return response


# Test the graph-enhanced agent
ga = GraphAgent()

inputs = [
    "Alice is the CTO of TechCorp. She manages three teams: Backend, Frontend, and Data.",
    "The Backend team is led by Charlie and they maintain the payment service and the user service.",
    "The Frontend team uses React and is led by Diana. They are building a new dashboard.",
    "What teams does Alice manage?",
    "Who leads the team that maintains the payment service?",
    "Tell me about the Backend team — who leads it and what do they work on?",
]

print("Graph-Enhanced Agent Demo")
print("=" * 60)

for user_input in inputs:
    print(f"\n👤 {user_input}")
    response = ga.step(user_input)
    print(f"🤖 {response[:300]}")

print(f"\n{'='*60}")
print(f"\n📊 Final graph: {ga.graph}")
print(ga.graph.visualize())

## 5. Vector vs. Graph — Head-to-Head Comparison

Let's systematically compare vector-based and graph-based memory on different query types. We'll test:

- **5 semantic queries** — finding conceptually related information (vector should win)
- **5 relational queries** — traversing relationships between entities (graph should win)

In [ ]:
# Build parallel vector and graph memories from the same facts
from dataclasses import dataclass

fact_texts = [
    "Alice is the CTO of TechCorp and has 15 years of experience",
    "Bob is the lead engineer at TechCorp working on the Atlas project",
    "The Atlas project uses Kubernetes for orchestration and PostgreSQL for storage",
    "PostgreSQL is a relational database that supports JSONB and full-text search",
    "Kubernetes runs on AWS EKS in the us-east-1 region",
    "Charlie manages the data science team at TechCorp",
    "The data science team uses PyTorch and runs experiments on GPU clusters",
    "TechCorp was founded in 2015 and is headquartered in San Francisco",
    "The Atlas project has a deadline of June 2025 and a budget of $2M",
    "Diana is the product manager for Atlas and reports to Alice",
]

# Vector memory
vector_index = faiss.IndexFlatIP(768)
vector_embeddings = embed_texts(fact_texts)
vector_index.add(np.array(vector_embeddings, dtype=np.float32))

def vector_query(question, k=3):
    q_emb = embed_texts([question])[0]
    scores, indices = vector_index.search(np.array([q_emb], dtype=np.float32), k)
    return [(float(scores[0][i]), fact_texts[indices[0][i]]) for i in range(k) if indices[0][i] >= 0]

# Graph memory
graph_mem = GraphMemory()
for text in fact_texts:
    graph_mem.add_memory(text)

print(f"Vector: {len(fact_texts)} facts indexed")
print(f"Graph: {graph_mem}")

# Semantic queries (vector should win)
semantic_queries = [
    "Tell me about database technologies",
    "What cloud infrastructure is used?",
    "Who has leadership experience?",
    "What machine learning tools are available?",
    "What are the project timelines and budgets?",
]

# Relational queries (graph should win)
relational_queries = [
    "Who reports to Alice?",
    "What project does Bob work on and what does it use?",
    "What technology stack does the Atlas project depend on?",
    "Who manages what teams at TechCorp?",
    "What is the chain from Alice to Kubernetes?",
]

print("\n" + "=" * 70)
print("VECTOR vs GRAPH COMPARISON")
print("=" * 70)

print("\n📊 SEMANTIC QUERIES (Vector's strength)")
print("-" * 70)
for q in semantic_queries:
    v_results = vector_query(q, k=2)
    g_results = graph_mem.query(q)

    print(f"\n  🔍 '{q}'")
    print(f"  Vector ({len(v_results)} results):")
    for score, fact in v_results[:2]:
        print(f"    [{score:.3f}] {fact[:65]}")
    print(f"  Graph ({len(g_results)} results):")
    for f in g_results[:2]:
        print(f"    {f['subject']} ──[{f['relation']}]──▶ {f['object']}")

print(f"\n\n📊 RELATIONAL QUERIES (Graph's strength)")
print("-" * 70)
for q in relational_queries:
    v_results = vector_query(q, k=2)
    g_results = graph_mem.query(q)

    print(f"\n  🔍 '{q}'")
    print(f"  Vector ({len(v_results)} results):")
    for score, fact in v_results[:2]:
        print(f"    [{score:.3f}] {fact[:65]}")
    print(f"  Graph ({len(g_results)} results):")
    for f in g_results[:2]:
        print(f"    {f['subject']} ──[{f['relation']}]──▶ {f['object']}")

print("\n" + "=" * 70)
print("✓ Vector excels at 'find me something about X' (semantic similarity)")
print("✓ Graph excels at 'how does X relate to Y?' (relationship traversal)")

## 6. Hybrid Memory — Best of Both Worlds

The ideal memory system **routes queries** to the right store:

- Semantic/conceptual queries → vector store
- Relational/structural queries → knowledge graph
- Ambiguous queries → both, then merge results

We build a `HybridMemory` that classifies queries and routes them appropriately.

In [ ]:
class HybridMemory:
    """Combines vector store and knowledge graph with query routing."""

    RELATION_KEYWORDS = {
        "who", "whom", "manages", "leads", "reports to", "works on",
        "depends on", "uses", "connects", "related to", "team",
        "between", "chain", "path", "hierarchy",
    }

    def __init__(self, embedding_dim=768):
        self.embedding_dim = embedding_dim
        # Vector store
        self.vector_index = faiss.IndexFlatIP(embedding_dim)
        self.vector_texts = []
        # Knowledge graph
        self.graph = GraphMemory()

    def store(self, text, metadata=None):
        """Store in both vector and graph."""
        # Vector store
        embedding = embed_texts([text])[0]
        self.vector_index.add(np.array([embedding], dtype=np.float32))
        self.vector_texts.append(text)

        # Graph store
        added, triples = self.graph.add_memory(text, metadata)
        return {"vector_stored": True, "graph_triples": len(triples), "graph_new": added}

    def classify_query(self, question):
        """Classify whether a query is semantic, relational, or both."""
        q_lower = question.lower()

        relational_score = sum(1 for kw in self.RELATION_KEYWORDS if kw in q_lower)
        has_question_word = any(q_lower.startswith(w) for w in ["who", "what", "which", "how"])

        if relational_score >= 2:
            return "relational"
        elif relational_score == 1 and has_question_word:
            return "both"
        elif any(kw in q_lower for kw in ["about", "describe", "explain", "tell me", "overview"]):
            return "semantic"
        else:
            return "both"

    def query(self, question, k=5):
        """Route query to appropriate memory store(s)."""
        query_type = self.classify_query(question)
        results = {"query_type": query_type, "vector_results": [], "graph_results": []}

        if query_type in ("semantic", "both"):
            if self.vector_index.ntotal > 0:
                q_emb = embed_texts([question])[0]
                n = min(k, self.vector_index.ntotal)
                scores, indices = self.vector_index.search(np.array([q_emb], dtype=np.float32), n)
                for score, idx in zip(scores[0], indices[0]):
                    if idx >= 0:
                        results["vector_results"].append({
                            "text": self.vector_texts[idx],
                            "similarity": float(score),
                            "source": "vector"
                        })

        if query_type in ("relational", "both"):
            graph_facts = self.graph.query(question)
            for f in graph_facts[:k]:
                results["graph_results"].append({
                    "triple": f"{f['subject']} {f['relation']} {f['object']}",
                    "source": "graph",
                    **f
                })

        return results

    def format_for_prompt(self, question, k=5):
        """Format query results for inclusion in an LLM prompt."""
        results = self.query(question, k)

        lines = [f"Memory recall (query type: {results['query_type']}):"]

        if results["vector_results"]:
            lines.append("  From semantic memory:")
            for r in results["vector_results"][:k]:
                lines.append(f"    [{r['similarity']:.2f}] {r['text'][:80]}")

        if results["graph_results"]:
            lines.append("  From knowledge graph:")
            for r in results["graph_results"][:k]:
                lines.append(f"    {r['triple']}")

        return "\n".join(lines)

    def stats(self):
        return {
            "vector_entries": len(self.vector_texts),
            "graph": self.graph.stats(),
        }

    def __repr__(self):
        s = self.stats()
        return (f"HybridMemory(vectors={s['vector_entries']}, "
                f"entities={s['graph']['entities']}, "
                f"relations={s['graph']['relations']})")


# Build and test hybrid memory
hybrid = HybridMemory()

for text in fact_texts:
    result = hybrid.store(text)

print(f"Hybrid Memory: {hybrid}")

# Test query routing
test_queries = [
    ("Tell me about databases and storage", "semantic"),
    ("Who reports to Alice?", "relational"),
    ("What technology does the Atlas project use?", "both"),
    ("Describe TechCorp's engineering culture", "semantic"),
    ("Who manages which teams?", "relational"),
]

print("\nQuery Routing Demo")
print("=" * 60)

for question, expected in test_queries:
    results = hybrid.query(question)
    actual = results["query_type"]
    match = "✅" if actual == expected else "⚠️"
    print(f"\n{match} '{question}'")
    print(f"   Routed to: {actual} (expected: {expected})")
    print(f"   Vector results: {len(results['vector_results'])}, Graph results: {len(results['graph_results'])}")

    formatted = hybrid.format_for_prompt(question, k=2)
    for line in formatted.split("\n"):
        print(f"   {line}")

## 7. Graph Maintenance — Contradictions, Redundancy, and Cleanup

Real-world knowledge graphs accumulate noise over time:

- **Contradictions**: "Alice is the CTO" vs. "Bob is the CTO" (same relation, different objects)
- **Redundancy**: "Alice leads the team" and "Alice is the team lead" (same meaning, different wording)
- **Staleness**: Facts that are no longer true (Alice was CTO, now Bob is)

We build maintenance routines to detect and handle these issues.

In [ ]:
class GraphMaintenance:
    """Tools for maintaining knowledge graph quality."""

    @staticmethod
    def find_contradictions(graph_memory):
        """Find potential contradictions: same subject+relation but different objects."""
        contradictions = []
        edge_groups = defaultdict(list)

        for s, o, data in graph_memory.graph.edges(data=True):
            relation = data.get("relation", "unknown")
            key = (s, relation)
            edge_groups[key].append(o)

        # Relations that should be unique (one subject → one object)
        unique_relations = {"is_cto_of", "is_ceo_of", "leads", "is_lead_of", "manages",
                           "headquartered_in", "founded_in", "uses_database", "deployed_on"}

        for (subject, relation), objects in edge_groups.items():
            if len(objects) > 1 and relation in unique_relations:
                contradictions.append({
                    "subject": subject,
                    "relation": relation,
                    "objects": objects,
                    "issue": f"Multiple values for unique relation: {subject} {relation} → {objects}"
                })

        return contradictions

    @staticmethod
    def find_redundancies(graph_memory, threshold=0.85):
        """Find semantically redundant triples using embeddings."""
        redundancies = []
        edges = list(graph_memory.graph.edges(data=True))
        if len(edges) < 2:
            return redundancies

        # Convert triples to text
        triple_texts = []
        for s, o, data in edges:
            r = data.get("relation", "related_to")
            triple_texts.append(f"{s} {r} {o}")

        # Embed and compare
        embeddings = embed_texts(triple_texts)
        for i in range(len(triple_texts)):
            for j in range(i + 1, len(triple_texts)):
                sim = cosine_similarity(embeddings[i], embeddings[j])
                if sim >= threshold:
                    redundancies.append({
                        "triple_a": triple_texts[i],
                        "triple_b": triple_texts[j],
                        "similarity": sim,
                    })

        return redundancies

    @staticmethod
    def graph_health_report(graph_memory):
        """Generate a health report for the knowledge graph."""
        g = graph_memory.graph
        stats = graph_memory.stats()

        # Connectivity
        if g.number_of_nodes() > 0:
            weakly_connected = nx.number_weakly_connected_components(g)
        else:
            weakly_connected = 0

        # Isolated nodes
        isolated = list(nx.isolates(g))

        # Degree distribution
        degrees = [d for _, d in g.degree()]
        avg_degree = sum(degrees) / len(degrees) if degrees else 0

        report = {
            "entities": stats["entities"],
            "relations": stats["relations"],
            "connected_components": weakly_connected,
            "isolated_nodes": len(isolated),
            "avg_degree": round(avg_degree, 2),
            "max_degree": max(degrees) if degrees else 0,
        }

        return report


# Demonstrate graph maintenance
maint_graph = GraphMemory()

# Add some facts including contradictions and redundancies
maint_facts = [
    "Alice is the CTO of TechCorp",
    "Bob is the CTO of TechCorp",       # contradiction with line 1
    "Alice leads the engineering team",
    "Alice is the leader of the engineering team",  # redundancy with line 3
    "Project Atlas uses PostgreSQL",
    "Project Atlas uses MySQL",          # potential contradiction
    "Charlie works in the data science department",
]

print("Graph Maintenance Demo")
print("=" * 60)

for fact in maint_facts:
    maint_graph.add_memory(fact)

# Health report
print("\n📋 Health Report:")
report = GraphMaintenance.graph_health_report(maint_graph)
for key, value in report.items():
    print(f"   {key}: {value}")

# Find contradictions
print("\n⚠️  Potential Contradictions:")
contradictions = GraphMaintenance.find_contradictions(maint_graph)
if contradictions:
    for c in contradictions:
        print(f"   {c['issue']}")
else:
    print("   None detected (relation names may not match unique set)")

# Find redundancies
print("\n🔄 Potential Redundancies:")
redundancies = GraphMaintenance.find_redundancies(maint_graph, threshold=0.80)
if redundancies:
    for r in redundancies:
        print(f"   [{r['similarity']:.3f}] '{r['triple_a']}' ≈ '{r['triple_b']}'")
else:
    print("   None detected above threshold")

print(f"\n{maint_graph.visualize()}")

## 8. Key Takeaways

### What We Built
1. **Entity-relation extraction** — LLM-based triple extraction from text
2. **GraphMemory** — networkx-based knowledge graph with entity context, multi-hop queries, and NL queries
3. **Graph-enhanced agent** — agent that builds and queries a knowledge graph during conversation
4. **Vector vs. graph comparison** — systematic evaluation showing complementary strengths
5. **HybridMemory** — combined vector + graph with automatic query routing
6. **Graph maintenance** — contradiction detection, redundancy finding, and health reports

### Key Insights

- **Vectors and graphs are complementary**, not competing — vectors find similar concepts, graphs traverse relationships
- **Multi-hop reasoning is graphs' killer feature** — "Who manages the team that uses PostgreSQL?" requires traversal, not similarity
- **Triple extraction is imperfect** — LLMs extract triples with varying quality; normalization and deduplication help
- **Hybrid routing improves recall** — sending queries to the right memory type avoids false negatives
- **Graph maintenance is essential** — contradictions and redundancy accumulate without active cleanup

### Design Decision Guide

| Need | Use |
|------|-----|
| "Find information about X" | Vector store |
| "How does X relate to Y?" | Knowledge graph |
| "What is connected to X within 3 hops?" | Knowledge graph |
| "Find documents similar to this text" | Vector store |
| "Who manages the team that built the service that uses the database?" | Knowledge graph |
| Production with mixed queries | Hybrid |

### What's Next
- **Notebook 13**: Advanced tool design — building robust, production-grade tools for agents
- **Notebook 14**: Code execution tool — agents that write and run code safely

### References
- Ji et al. (2022) — *"A Survey of Knowledge Graphs: Representation, Acquisition, and Applications"*
- Pan et al. (2024) — *"Unifying Large Language Models and Knowledge Graphs: A Roadmap"*
- Park et al. (2023) — *"Generative Agents"* (used relationship-based memory retrieval)